In [ ]:
from langgraph.graph import StateGraph, START, END
from groq import Groq
from typing import TypedDict

In [ ]:
import os
from dotenv import load_dotenv

# Load variables from .env into environment
load_dotenv()

# Get the key
api_key = os.getenv("API_KEY")

# Ensure the key exists
if not api_key:
    raise ValueError("❌ API_KEY not found in .env file")


client = Groq(api_key=api_key)



In [ ]:
class LLMState(TypedDict):
    question:str
    question2:str
    answer:str

In [ ]:
def contradictor(state: "LLMState") -> "LLMState":

    # Extract topic
    topic = state["question"]

    # Call Groq Chat Completion
    response = client.chat.completions.create(
        model="mixtral-8x7b-32768",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful AI that generates deep contradictory reasoning questions."
                )
            },
            {
                "role": "user",
                "content": f"Create a contradictory question for this topic:\n{topic}"
            }
        ],
        temperature=0.7
    )

    # Extract output safely
    generated_question = response.choices[0].message.content.strip()

    # Update state
    state["question2"] = generated_question

    return state


In [ ]:


def prompt_creator(state: "LLMState") -> "LLMState":
    # Extract topic
    topic = state["question"]

    # Call Groq Chat Completion
    response = client.chat.completions.create(
        model="mixtral-8x7b-32768",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a great prompt engineer"
                )
            },
            {
                "role": "user",
                "content": f"generate a better prompt for this :\n{topic}"
            }
        ],
        temperature=0.7
    )

    # Extract output safely
    generated_question = response.choices[0].message.content.strip()

    # Update state
    state["question"] = generated_question

    return state


In [ ]:
def llmresponse(state: "LLMState") -> "LLMState":
    # Extract topic
    question = state["question"]
    question2 = state["question2"]

    # Call Groq Chat Completion
    response = client.chat.completions.create(
        model="mixtral-8x7b-32768",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a good reasoning assistant"
                )
            },
            {
                "role": "user",
                "content": f"answer the question {question}, along with it, also answer it's contradictory question {question2}"
            }
        ],
        temperature=0.7
    )

    # Extract output safely
    generated_answer = response.choices[0].message.content.strip()

    # Update state
    state["answer"] = generated_answer

    return state


In [ ]:
graph = StateGraph(LLMState)
graph.add_node('llmresponse',llmresponse)
graph.add_node('prompt_creator',prompt_creator)
graph.add_node('contradictor',contradictor)

In [ ]:
graph.add_edge(START,'prompt_creator')
graph.add_edge(START,'contradictor')
graph.add_edge('prompt_creator','llmresponse')
graph.add_edge('contradictor','llmresponse')
graph.add_edge('llmresponse',END)


In [ ]:
graph.compile()